# Mašinsko učenje - Vežba 1

Ulazni podaci:

In [1]:
import numpy as np

X = np.array([520, 610, 700, 820, 910,
              1030, 1120, 1250, 1390, 1510], dtype=float)

Y = np.array([3.1, 3.8, 4.0, 5.1, 5.3,
              6.0, 6.2, 5.9, 5.1, 4.1], dtype=float)

## Zadatak 1

##### Sekcije 1.1, 1.2 i 1.3 koji se odnose na izvođenja, kao i prvi deo 1.4 u prvom zadatku priloženi su kao poseban pdf dokument jer su rađeni na papiru.

##### 1.4  Dobijanje vektora parametra w korišćenjem numpy.linalg.solve - primer

In [3]:
import numpy as np

X = np.array([520, 610, 700, 820, 910,
              1030, 1120, 1250, 1390, 1510])

Y = np.array([3.1, 3.8, 4.0, 5.1, 5.3,
              6.0, 6.2, 5.9, 5.1, 4.1])

M = 2

Phi = np.vander(X, N=M+1, increasing=True)

A = Phi.T @ Phi
b = Phi.T @ Y

w = np.linalg.solve(A, b)

print(w)

[ 5.08946922e+00 -6.64469449e-05 -1.52898007e-07]


- `np.vander(X, N=M+1, increasing=True)` formira regresionu matricu sa kolonama:

  $$1,\ x,\ x^2,\ \ldots,\ x^M$$

- `Phi.T` predstavlja transponovanu matricu:

  $$\Phi^\top$$

- `@` označava množenje matrica.

- `A` predstavlja matricu:

  $$A = \Phi^\top \Phi$$

- `b` predstavlja vektor:

  $$\mathbf{b} = \Phi^\top \mathbf{y}$$

- `np.linalg.solve(A, b)` rešava sistem:

  $$A\mathbf{w} = \mathbf{b}$$

Ovaj pristup zahteva da matrica $A$ bude invertibilna.



##### 1.4  Dobijanje vektora parametra w korišćenjem numpy.linalg.lstsq - primer

In [4]:
w, residuals, rank, s = np.linalg.lstsq(Phi, Y, rcond=None)

print(w)

[-5.28943056e+00  2.04421901e-02 -9.33187438e-06]


 Ovde ne moramo eksplicitno da formiramo matricu $\Phi^\top\Phi$.

Funkcija np.linalg.lstsq traži vektor parametara koji minimizuje:
$$
\boxed{\|\Phi\mathbf{w}-\mathbf{y}\|_2^2}
$$
Ova funkcija je pogodna i kada matrica nema pun rang, odnosno kada sistem nema jedinstveno rešenje.

*Ova dva primera koriste iste podatke, ali različite numeričke postupke. Za realne podatke je obično bolje direktno koristiti np.linalg.lstsq, naročito ako je stepen polinoma veći.*

1.5. deo prvog zadatka takođe je rešen na papiru


## Zadatak 2 : Implementacija u scikit-learn

U ovom zadatku implementira se klasa `PolynomialRegression`, koja omogućava polinomsku regresiju i kompatibilna je sa bibliotekom `scikit-learn`.

Klasa sadrži tri metode:
- fit(X, y): određuje koeficijente polinoma rešavanjem normalnih jednačina.
- predict(X): izračunava predikcije za zadate vrednosti promenljive \(X\).
- score(X, y): računa srednju kvadratnu grešku (MSE) između stvarnih i predviđenih vrednosti.

In [34]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin

Klasa:

In [35]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin


class PolynomialRegression(BaseEstimator, RegressorMixin):

    def __init__(self, degree=1):
        self.degree = degree
        self.coef_ = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float).reshape(-1)
        y = np.asarray(y, dtype=float).reshape(-1)

        Phi = np.vander(X, N=self.degree + 1, increasing=True)

        A = Phi.T @ Phi
        b = Phi.T @ y

        self.coef_ = np.linalg.solve(A, b)

        return self

    def predict(self, X):

        X = np.asarray(X, dtype=float).reshape(-1)

        Phi = np.vander(X, N=self.degree + 1, increasing=True)

        return Phi @ self.coef_

    def score(self, X, y):

        y = np.asarray(y).reshape(-1)
        y_pred = self.predict(X)

        return np.mean((y - y_pred) ** 2)

Metode `BaseEstimator` i `RegressorMixin` preuzete iz biblioteke `scikit-learn` omogućavaju da se naša klasa uklopi u standardni način rada ove biblioteke. Parametar `degree` predstavlja stepen polinoma koji želimo da koristimo. Na primer, za stepen polinoma 2 model određuje tri koeficijenta: \(w_0\), \(w_1\) i \(w_2\).

Funkcija `np.vander()` koristi se za formiranje Vandermondove matrice, čije kolone predstavljaju stepene ulazne promenljive \(X\), počevši od nultog stepena. Na osnovu tako formirane matrice izračunavaju se elementi sistema normalnih jednačina. Matrica `A` predstavlja proizvod transponovane Vandermondove matrice i same matrice, dok vektor `b` predstavlja proizvod transponovane Vandermondove matrice i vektora ciljnih vrednosti `y`.

Za rešavanje dobijenog sistema koristi se funkcija `np.linalg.solve()`, koja određuje vrednosti koeficijenata polinoma. Izračunati koeficijenti čuvaju se u atributu `self.coef_`, kako bi mogli da se koriste u narednim metodama klase. Metoda `fit` na kraju vraća objekat `self`, odnosno istrenirani model.

Metoda `predict` koristi izračunate koeficijente i Vandermondovu matricu kako bi odredila predviđene vrednosti za zadate ulazne podatke. Na taj način model može da vrši predikcije i za vrednosti koje nisu korišćene tokom treniranja.

Metoda `score` izračunava srednju kvadratnu grešku (MSE), odnosno prosečnu vrednost kvadrata razlika između stvarnih i predviđenih vrednosti. Manja vrednost MSE ukazuje na manje odstupanje predikcija od stvarnih vrednosti.

Učitavanje podataka:

In [36]:
import pandas as pd

df = pd.read_csv("podaci.csv", sep=";", decimal=",")

X = df["X"].astype(float).values
y = df["Y"].astype(float).values

In [37]:
print(df.head())
print(X)
print(y)

     X    Y
0  520  3.1
1  610  3.8
2  700  4.0
3  820  5.1
4  910  5.3
[ 520.  610.  700.  820.  910. 1030. 1120. 1250. 1390. 1510.]
[3.1 3.8 4.  5.1 5.3 6.  6.2 5.9 5.1 4.1]


*Testiranje implementirane klase*

Nakon definisanja klase `PolynomialRegression`, potrebno je proveriti njen rad na učitanom skupu podataka. Za testiranje se koristi polinom drugog stepena (\(M=2\)). Najpre se formira model, a zatim se poziva metoda `fit`, kojom se određuju koeficijenti polinoma. Pomoću metode `predict` dobijaju se predviđene vrednosti, dok se metodom `score` izračunava srednja kvadratna greška (MSE).

In [38]:
M = 2

model = PolynomialRegression(degree=M)

model.fit(X, y)

y_pred = model.predict(X)

mse = model.score(X, y)


print("Koeficijenti w:", model.coef_)
print("Srednja kvadratna greska:", mse)

Koeficijenti w: [-5.28943056e+00  2.04421901e-02 -9.33187438e-06]
Srednja kvadratna greska: 0.05755136383219543


Na osnovu dobijenih rezultata možemo videti da je model uspešno odredio koeficijente polinoma drugog stepena. Vrednosti koeficijenata ukazuju na to da linearni član ima veći uticaj na model u odnosu na kvadratni član, čija je vrednost znatno manja. Srednja kvadratna greška (MSE) iznosi približno 0.05755, što predstavlja meru odstupanja predviđenih od stvarnih vrednosti. Dobijeni rezultati potvrđuju da implementirana klasa pravilno funkcioniše i da može da se koristi za polinomsku regresiju.

## Zadatak 3 : Polinomska regresija za različite stepene polinoma
U ovom zadatku ispituje se uticaj stepena polinoma na rezultate polinomske regresije. Modeli se treniraju za stepene polinoma 1, 2, 3, 5 i 9. Za svaki model određuju se koeficijenti polinoma, predviđene vrednosti i srednja kvadratna greška (MSE), kako bi se uporedila uspešnost prilagođavanja modela datim podacima.

Kod:

In [39]:
for M in [1, 2, 3, 5, 9]:

    model = PolynomialRegression(degree=M)
    model.fit(X, y)

    y_pred = model.predict(X)
    mse = model.score(X, y)

    print(f"Stepen polinoma M = {M}")
    print("Koeficijenti w:", model.coef_)
    print("MSE:", mse)
    print("----------------------------------")

Stepen polinoma M = 1
Koeficijenti w: [3.28955486e+00 1.59274355e-03]
MSE: 0.7484529689123315
----------------------------------
Stepen polinoma M = 2
Koeficijenti w: [-5.28943056e+00  2.04421901e-02 -9.33187438e-06]
MSE: 0.05755136383219543
----------------------------------
Stepen polinoma M = 3
Koeficijenti w: [ 2.33050642e+00 -5.26280508e-03  1.76031157e-05 -8.85852296e-09]
MSE: 0.01655313298904368
----------------------------------
Stepen polinoma M = 5
Koeficijenti w: [-1.74656509e+01  1.15336343e-01 -2.64477828e-04  3.07695644e-07
 -1.70703361e-10  3.54804378e-14]
MSE: 0.011185122490545105
----------------------------------
Stepen polinoma M = 9
Koeficijenti w: [-2.10510179e+03  1.54794216e+01 -4.17909944e-02  3.68605992e-05
  5.35138507e-08 -1.75887992e-10  2.04308248e-13 -1.25489912e-16
  4.05693620e-20 -5.45172884e-24]
MSE: 0.004975948606236068
----------------------------------


Primećujemo da se MSE smanjuje sa povećanjem stepena polinoma. Međutim, to samo pokazuje da se složeniji modeli bolje prilagođavaju podacima za treniranje. Kod polinoma devetog stepena mogu se javiti problemi sa preprilagođavanjem (overfitting), što ćemo dodatno proveriti kroz predikcije novih vrednosti.

#### Predvdjanje za nove ulazne vrednosti

In [40]:
X_1 = np.array([600, 900, 1300, 1600])

for M in [1, 2, 3, 5, 9]:

    model1 = PolynomialRegression(degree=M)
    model1.fit(X, y)

    print(f"predikcija za M={M} :", model1.predict(X_1))

predikcija za M=1 : [4.24520099 4.72302405 5.36012147 5.83794454]
predikcija za M=2 : [3.61640874 5.5497223  5.51454889 3.52847522]
predikcija za M=3 : [3.59650407 5.39464234 5.77595043 2.68948448]
predikcija za M=5 : [3.62219936 5.37251059 5.70190913 3.64842651]
predikcija za M=9 : [  3.78726743   5.39864563   5.40872579 -22.53618726]


Na osnovu dobijenih rezultata možemo da vidimo da modeli polinoma stepena od 1 do 5 daju relativno stabilne i realne predikcije za nove vrednosti ulazne promenljive. Sa druge strane, kod polinoma devetog stepena primećujemo da predikcija za vrednost X = 1600 značajno odstupa od ostalih rezultata i poprima negativnu vrednost, što nije realno očekivano za biomasu.

Ovakvo ponašanje ukazuje na nestabilnost polinoma visokog stepena, naročito pri ekstrapolaciji, odnosno predviđanju vrednosti izvan opsega podataka korišćenih za treniranje. Zbog toga složeniji polinom ne mora nužno da predstavlja bolji izbor, čak i kada ostvaruje manju grešku na podacima za treniranje.

## Zadatak 4 : Proširenje klase PolynomialRegression

Proširenje klase PolynomialRegression parametrom za L2 regularizaciju

In [42]:
class PolynomialRegression(BaseEstimator, RegressorMixin):

    def __init__(self, degree=1, reg_lambda=0.0):
        self.degree = degree
        self.reg_lambda = reg_lambda
        self.coef_ = None

    def fit(self, X, y):

        X = np.asarray(X, dtype=float).reshape(-1)
        y = np.asarray(y, dtype=float).reshape(-1)

        Phi = np.vander(X, N=self.degree + 1, increasing=True)

        A = Phi.T @ Phi + self.reg_lambda * np.eye(Phi.shape[1])
        b = Phi.T @ y

        self.coef_ = np.linalg.solve(A, b)

        return self

    def predict(self, X):

        X = np.asarray(X, dtype=float).reshape(-1)

        Phi = np.vander(X, N=self.degree + 1, increasing=True)

        return Phi @ self.coef_

    def score(self, X, y):

        y = np.asarray(y, dtype=float).reshape(-1)
        y_pred = self.predict(X)

        return np.mean((y - y_pred) ** 2)

Poređenje:

In [49]:
M = 5

# BEZ REGULARIZACIJE

model_bez = PolynomialRegression(degree=M, reg_lambda=0.0)
model_bez.fit(X, y)

y_pred_bez = model_bez.predict(X)
mse_bez = model_bez.score(X, y)

print("Bez regularizacije:")
print(f"Stepen polinoma M = {M}")
print("Koeficijenti w:", model_bez.coef_)
print("MSE:", mse_bez)
print("\n\n\n")

# SA REGULARIZACIJOM

model_sa = PolynomialRegression(degree=M, reg_lambda=10.0)
model_sa.fit(X, y)

y_pred_sa = model_sa.predict(X)
mse_sa = model_sa.score(X, y)

print("Sa regularizacijom:")
print(f"Stepen polinoma M = {M}")
print("Koeficijenti w:", model_sa.coef_)
print("MSE:", mse_sa)

Bez regularizacije:
Stepen polinoma M = 5
Koeficijenti w: [-1.74656509e+01  1.15336343e-01 -2.64477828e-04  3.07695644e-07
 -1.70703361e-10  3.54804378e-14]
MSE: 0.011185122490545105




Sa regularizacijom:
Stepen polinoma M = 5
Koeficijenti w: [ 3.60250417e-05  1.71152892e-02 -5.18366795e-05  8.56844025e-08
 -5.86048702e-11  1.35168557e-14]
MSE: 0.012237013152262811


Na osnovu dobijenih rezultata možemo da primetimo da je nakon uvođenja L2 regularizacije došlo do promene vrednosti koeficijenata polinoma. Njihove vrednosti su uglavnom smanjene, što je posledica dodatog parametra regularizacije λ = 10.

Takođe, primećujemo da se srednja kvadratna greška povećala sa 0.01119 na 0.01224. To znači da model sa regularizacijom nešto slabije prati podatke korišćene za treniranje. Ipak, ograničavanjem vrednosti koeficijenata model postaje manje sklon preteranom prilagođavanju podacima (overfitting).

##### Kako se regularizacija odražava na predikcije novih vrednosti za različite stepene polinoma: M = 1, 2, 3, 5 i 9

In [48]:
X_1 = np.array([600, 900, 1300, 1600])

for M in [1, 2, 3, 5, 9]:

    # BEZ REGULARIZACIJE

    model_bez = PolynomialRegression(degree=M, reg_lambda=0.0)
    model_bez.fit(X, y)

    mse_bez = model_bez.score(X, y)
    y_pred_bez = model_bez.predict(X_1)

    # SA REGULARIZACIJOM

    model_sa = PolynomialRegression(degree=M, reg_lambda=10.0)
    model_sa.fit(X, y)

    mse_sa = model_sa.score(X, y)
    y_pred_sa = model_sa.predict(X_1)

    print(f"Stepen polinoma M = {M}")
    print("MSE bez regularizacije:", mse_bez)
    print("Predikcije bez regularizacije:", y_pred_bez)
    print("MSE sa regularizacijom:", mse_sa)
    print("Predikcije sa regularizacijom:", y_pred_sa)
    print("\n\n\n")

Stepen polinoma M = 1
MSE bez regularizacije: 0.7484529689123315
Predikcije bez regularizacije: [4.24520099 4.72302405 5.36012147 5.83794454]
MSE sa regularizacijom: 1.593510670275254
Predikcije sa regularizacijom: [2.89644241 4.20422487 5.94793481 7.25571727]




Stepen polinoma M = 2
MSE bez regularizacije: 0.05755136383219543
Predikcije bez regularizacije: [3.61640874 5.5497223  5.51454889 3.52847522]
MSE sa regularizacijom: 0.2921771963441442
Predikcije sa regularizacijom: [4.18046566 5.17309286 5.33515212 4.5856138 ]




Stepen polinoma M = 3
MSE bez regularizacije: 0.01655313298904368
Predikcije bez regularizacije: [3.59650407 5.39464234 5.77595043 2.68948448]
MSE sa regularizacijom: 0.02009028227153851
Predikcije sa regularizacijom: [3.58292274 5.45120984 5.70827885 2.89033962]




Stepen polinoma M = 5
MSE bez regularizacije: 0.011185122490545105
Predikcije bez regularizacije: [3.62219936 5.37251059 5.70190913 3.64842651]
MSE sa regularizacijom: 0.012237013152262811
Predikcije 

Na osnovu dobijenih rezultata možemo da primetimo da je MSE veći kod modela sa L2 regularizacijom za sve ispitivane stepene polinoma. To znači da regularizovani modeli nešto slabije prilagođavaju trening podacima u odnosu na modele bez regularizacije.

Međutim, najveća razlika uočava se kod polinoma devetog stepena. Bez regularizacije, za ulaznu vrednost X = 1600 dobijamo negativnu predikciju, što nije realno očekivano za posmatranu veličinu. Sa druge strane, nakon uvođenja regularizacije dobijamo predikciju 3.2915, koja je znatno realističnija.

Ovi rezultati pokazuju da L2 regularizacija može doprineti stabilnijem ponašanju modela i smanjenju preteranog prilagođavanja trening podacima (overfitting), posebno kod polinoma višeg stepena. Ipak, povećanje parametra regularizacije može dovesti i do većeg odstupanja na trening skupu, zbog čega je potrebno pronaći odgovarajuću vrednost parametra λ.